1. Error Handling

Fetch data from the JSONPlaceholder API ("https://jsonplaceholder.typicode.com/end-point"). Implement robust error handling to manage scenarios where:
The API endpoint is invalid.
The API returns a non-success status code.
The network connection fails or times out.
Log all errors to a file named api_errors.log with details such as the timestamp, error type, and error message.

In [1]:
import requests
import logging
import psycopg2
from dotenv import load_dotenv
import os

filelogger = logging.getLogger(__name__)
filelogger.level = logging.INFO
handler = logging.FileHandler('api_errors.log', mode='a')
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
filelogger.addHandler(handler)

try:

    response = requests.get("https://jsonplaceholder.typicode.com/users",timeout=5)


    print(response.status_code)
    print(response.raise_for_status())

except requests.exceptions.Timeout as e:
        filelogger.error(f'Timeout Error :- {str(e)}')
        print('Request Timeout Error')

except requests.exceptions.ConnectionError as e:
        filelogger.error(f'Connection Error :- {str(e)}')
        print('Network connection Failed')

except requests.exceptions.RequestException as e:
        logging.error(f"Request Exception :- {str(e)}")
        print('General request error occurred')

except Exception as e:
        filelogger.error(f'Unexpected Error Occurred :- {str(e)}')
        print('Unexpected Error Occurred')

200
None


2. Database Storage with PostgreSQL

Set up a PostgreSQL database named api_database with the following tables:
Users api endpoint: https://jsonplaceholder.typicode.com/users
Users table should contain following columns: id, name, email, address (formatted as a single string).
Posts api endpoint: https://jsonplaceholder.typicode.com/posts
Posts table should contain following columns: id, user_id, title, body.
Use Python's psycopg2 or sqlalchemy library to connect to the database.
Fetch and store user data from the API's /users endpoint into the Users table.
Fetch and store post data from the API's /posts endpoint into the Posts table.
Ensure duplicate entries are avoided by using ON CONFLICT DO NOTHING or similar techniques.

In [2]:
import psycopg2

conn = psycopg2.connect(
    host=os.getenv('DB_HOST'),
    database=os.getenv('DB_DATABASE'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD')
)


cursor = conn.cursor()

In [3]:
import requests

response = requests.get("https://jsonplaceholder.typicode.com/users",timeout=5)
print(response.status_code)
 
user = response.json()


for i in user:
    print(i)

200
{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}
{'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'address': {'street': 'Victor Plains', 'suite': 'Suite 879', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'geo': {'lat': '-43.9509', 'lng': '-34.4618'}}, 'phone': '010-692-6593 x09125', 'website': 'anastasia.net', 'company': {'name': 'Deckow-Crist', 'catchPhrase': 'Proactive didactic contingency', 'bs': 'synergize scalable supply-chains'}}
{'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'address': {'street': 'Douglas Exte

In [4]:
for i in user:

    address = f"{i['address']['street']}, {i['address']['city']}"

    cursor.execute("""
    INSERT INTO users(id,name,email,address) values (%s, %s, %s, %s)
    ON CONFLICT(id) DO NOTHING """,
    (i['id'],i['name'],i['email'],address)
    )

conn.commit()

In [5]:
import requests

response = requests.get("https://jsonplaceholder.typicode.com/posts",timeout=5)
print(response.status_code)
 
posts = response.json()


for i in posts:
    print(i)

200
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}
{'userId': 1, 'id': 2, 'title': 'qui est esse', 'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'}
{'userId': 1, 'id': 3, 'title': 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'body': 'et iusto sed quo iure\nvoluptatem occaecati omnis eligendi aut ad\nvoluptatem doloribus vel accusantium quis pariatur\nmolestiae porro eius odio et labore et velit aut'}
{'userId': 1, 'id': 4, 'title': 'eum et est occaecati', 'body': 'ullam et saepe reiciendis voluptatem adipisci\nsit amet autem assumenda provident rerum culpa\nquis hic c

In [6]:
for i in posts:
    cursor.execute("""
    INSERT INTO posts(id,user_id,title,body) values (%s, %s, %s, %s)
    ON CONFLICT(id) DO NOTHING """,
    (i['id'],i['userId'],i['title'],i['body'])
    )

conn.commit()

3. Scheduled API Calls

Use the schedule or APScheduler library to run a task every 10 minutes. The task should:
Fetch new posts from the /posts endpoint.
Insert only new posts into the database.
Log each execution with the timestamp and the number of new posts added to the database.
If an error occurs during the scheduled task (e.g., the API is unavailable), log the error to api_errors.log.


4. Summary Report

After each scheduled task execution, print a summary report showing:
The total number of users in the database.
The total number of posts in the database.
The number of new posts added during the most recent execution.
Ensure the script runs indefinitely until manually stopped.

In [7]:
import requests
import schedule
import time
import logging



def fetch_posts():

    new_posts = 0

    try:
        url = "https://jsonplaceholder.typicode.com/posts"
        response = requests.get(url, timeout=5)
        response.raise_for_status()

        posts = response.json()

        conn = psycopg2.connect(
                 host=os.getenv('DB_HOST'),
                 database=os.getenv('DB_DATABASE'),
                 user=os.getenv('DB_USER'),
                 password=os.getenv('DB_PASSWORD')
)

        cursor = conn.cursor()

        for post in posts:

            cursor.execute("""
                INSERT INTO posts (id, user_id, title, body)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT (id) DO NOTHING
            """, (
                post["id"],
                post["userId"],
                post["title"],
                post["body"]
            ))

            if cursor.rowcount > 0:
                new_posts += 1

        conn.commit()

        cursor.close()
        conn.close()

        filelogger.info(f"Scheduler executed. {new_posts} new posts added.")

        return new_posts

    except Exception as e:
        filelogger.error(f"Scheduled task error: {e}")
        return 0

In [8]:
def print_summary(new_posts):

    conn = psycopg2.connect(
          host=os.getenv('DB_HOST'),
          database=os.getenv('DB_DATABASE'),
          user=os.getenv('DB_USER'),
          password=os.getenv('DB_PASSWORD') )

    cursor = conn.cursor()

    cursor.execute("SELECT COUNT(*) FROM users")
    total_users = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM posts")
    total_posts = cursor.fetchone()[0]

    cursor.close()
    conn.close()

    print(f"Total Users: {total_users}")
    print(f"Total Posts: {total_posts}")
    print(f"New Posts Added: {new_posts}")

In [9]:
def scheduled_job():

    new_posts = fetch_posts()
    print_summary(new_posts)

schedule.every(10).seconds.do(scheduled_job)

while True:

    schedule.run_pending()


Total Users: 10
Total Posts: 100
New Posts Added: 0
Total Users: 10
Total Posts: 100
New Posts Added: 0
Total Users: 10
Total Posts: 100
New Posts Added: 0
Total Users: 10
Total Posts: 100
New Posts Added: 0


KeyboardInterrupt: 